# Part 1


In [101]:
import pandas as pd

## Task 1


In [102]:
bike_rentals_raw = pd.read_csv("../bike_rentals.csv")
bike_rentals = bike_rentals_raw.copy()

Shape should be (403, 15). 'text' is 'str'.

In [103]:
print(bike_rentals.shape)
print(bike_rentals.dtypes)

(403, 15)
rental_id            str
station              str
bike_type            str
helmets_text         str
is_member            str
insured              str
paid_online          str
status               str
coupon_code      float64
damage_notes     float64
late_fee_note    float64
start_date           str
end_date             str
price                str
rating           float64
dtype: object


Columns where every value is missing. Should be coupon_code, damage_notes and late_fee_note.

In [104]:
missing_columns = bike_rentals.columns[bike_rentals.isna().all()]

print(list(missing_columns))

['coupon_code', 'damage_notes', 'late_fee_note']


Result after removal of empty columns. Should give (403, 12) and 0 empty columns.

In [105]:
#Do the removal, and then print it out.
bike_rentals = bike_rentals.dropna(axis=1, how='all')

print(bike_rentals.shape)

(403, 12)


In [106]:
missing_columns_after = bike_rentals.columns[bike_rentals.isna().all()]
print(list(missing_columns_after))
print(bike_rentals.isna().all().sum())

[]
0


## Task 2

In [107]:
#Looking at the first rows of 'price'.
print(bike_rentals['price'].head(3))

0      SEK 855.50
1    SEK 1,311.50
2      SEK 350.00
Name: price, dtype: str


Removal of distracting parts for further processing in the 'price' column.

In [108]:
price = bike_rentals['price'].str.replace("SEK ", "")
print(price.head(3).tolist())

#Thousands marker removal.
price = price.str.replace(",", "")
print(price.head(3).tolist())

['855.50', '1,311.50', '350.00']
['855.50', '1311.50', '350.00']


Should be float64 when described. Values of interest: 
count : 352.0
mean : 847.3
std : 633.4
min : 154.5
median : 668.2 / 668.3
max : 3401.0

In [109]:
bike_rentals['price'] = price.astype(float)

print(bike_rentals['price'].dtype)
print(bike_rentals['price'].describe().round(1))

float64
count     352.0
mean      847.3
std       633.4
min       154.5
25%       406.0
50%       668.2
75%      1194.6
max      3401.0
Name: price, dtype: float64


Top 3 with specific values given. Should be:

187 - R-5282 | Cargo bike | Completed | 3401.0

49 - R-5209 | Cargo bike | Completed | 2979.5

270 - R-5378 | E-bike | Completed | 2941.0

In [110]:
bike_rentals.sort_values('price', ascending=False).head(3)[['rental_id', 'bike_type', 'status', 'price']]

,rental_id,bike_type,status,price
187,R-5282,Cargo bike,Completed,3401.0
49,R-5209,Cargo bike,Completed,2979.5
270,R-5378,E-bike,Completed,2941.0


## Task 3

Missing amount and percentage should be 51 and 12.7%, respectively.

In [111]:
print(bike_rentals['price'].isna().sum())
print((bike_rentals['price'].isna().mean() * 100).round(1))

51
12.7


[No price] x [Cancelled rental] : Crosstab
The 4 not cancelled without price are missing without any clear reason, likely unintentional.
The 47 cancelled without price are missing because they weren't paid for.

In [112]:
no_price = bike_rentals['price'].isna()
cancelled_rental = bike_rentals['status'] == 'Cancelled'

pd.crosstab(no_price, cancelled_rental)

status,False,True
price,,
False,352,0
True,4,47


Leaving them to be as they are for the moment; no greater incentive yet to remove any of these.

## Task 4

Columns containing 't' and 'f' (Boolean representation).
Should be 'is_member', 'insured' and 'paid_online'. 'status' will not be here, because it is not represented binarily. 
It has a given status message.

In [113]:
tf_columns = [col for col in bike_rentals.columns if set(bike_rentals[col].dropna().unique()) <= {'t', 'f'}]

print(tf_columns)

['is_member', 'insured', 'paid_online']


Mapping.

In [114]:
bike_rentals['is_member'] = bike_rentals['is_member'].map({'t' : True, 'f' : False})
bike_rentals['insured'] = bike_rentals['insured'].map({'t' : True, 'f' : False})
bike_rentals['paid_online'] = bike_rentals['paid_online'].map({'t' : True, 'f' : False})

3 missing on insured, 0 on both is_member and paid_online.

In [115]:
print(bike_rentals[['is_member', 'insured', 'paid_online']].isna().sum())

is_member      0
insured        3
paid_online    0
dtype: int64


115 members, 288 non-members.

In [116]:
print(bike_rentals['is_member'].sum())
print((bike_rentals['is_member'] == False).sum())

115
288


Median comparison of price : Members vs. Non-members (+ N.o. rentals).
[248 count, 756 median] vs. [104 count, 558 median]
Members generally pay less.

In [117]:
bike_rentals.groupby('is_member')['price'].agg(['count', 'median']).round(0)

,count,median
is_member,,
False,248,756.0
True,104,558.0


## Task 5

0 missing start and end dates. (Before)

2 missing start dates, 0 missing end dates. (After)

Types of these two should now be datetime64[us].

In [118]:
date_columns = ['start_date', 'end_date']

before = bike_rentals[date_columns].isna().sum()

for col in date_columns:
    bike_rentals[col] = pd.to_datetime(bike_rentals[col], errors='coerce')

after = bike_rentals[date_columns].isna().sum()

print(before)
print(after)
print(bike_rentals[date_columns].dtypes)

start_date    0
end_date      0
dtype: int64
start_date    2
end_date      0
dtype: int64
start_date    datetime64[us]
end_date      datetime64[us]
dtype: object


In [ ]:
bike_rentals['rental_days'] = (bike_rentals['end_date'] - bike_rentals['start_date']).dt.days

print(bike_rentals['rental_days'].describe().round(0))

count    401.0
mean       3.0
std        2.0
min       -2.0
25%        1.0
50%        3.0
75%        4.0
max        7.0
Name: rental_days, dtype: float64


The minimum field will show a negative value. If the difference is negative, the end_date must be earlier than the start_date, which normally wouldn't make sense.

## Task 6

115 missing ratings.

In [123]:
print(bike_rentals['rating'].isna().sum())

115


Crosstab. 47 cancelled, 46 completed, 22 no show. Total = 115.

In [132]:
no_rating = bike_rentals['rating'].isna()

print(no_rating.sum())

pd.crosstab(no_rating, bike_rentals['status'])

115


status,Cancelled,Completed,No-show
rating,,,
False,0,288,0
True,47,46,22


There is no perfect match, because the 115 missing ratings don't correlate to one single status.

[has_rating] - True: 288, False: 115
Median: 5.00, mean: 4.23

In [ ]:
bike_rentals['has_rating'] = bike_rentals['rating'].notna()

print(bike_rentals['has_rating'].value_counts())
bike_rentals['rating'].describe().round(2)

has_rating
True     288
False    115
Name: count, dtype: int64


count    288.00
mean       4.23
std        1.02
min        1.00
25%        4.00
50%        5.00
75%        5.00
max        5.00
Name: rating, dtype: float64

## Task 7